# Fine-tune the policy-tone model (Colab)

**Before you run anything**
1. Runtime menu -> *Change runtime type* -> **T4 GPU** -> Save.
2. Run the cells from top to bottom (Shift+Enter). The login cell asks for a Hugging Face token with **write** access.
3. Change `TASK` in the config cell to `"topic"` and run again later if you want the topic model too.

What this does: trains `ProsusAI/finbert` to predict the **weak labels** (the lexicon's labels) on sentences up to 2018, picks the best epoch on 2019-2021, and reports on 2022-2026.
**Those scores measure agreement with the rules, not truth.** The real test is the hand-labelled gold set, scored separately.
The model and its predictions are saved **privately**.

In [ ]:
!pip -q install transformers huggingface_hub scikit-learn

In [ ]:
TASK = "tone"                      # "tone" or "topic"
BASE_MODEL = "ProsusAI/finbert"
USER = "Kobichris"
DATA_REPO = f"{USER}/cb-policy-tone-data"   # private dataset made by `python -m cb_policy_tone.dataset --push`
MODEL_REPO = f"{USER}/cb-policy-{TASK}"
MAX_LEN, BATCH, EPOCHS, LR, SEED = 128, 32, 4, 2e-5, 42

LABELS = {
    "tone": ["dovish", "neutral", "hawkish"],
    "topic": ["inflation", "growth", "external", "fiscal", "financial", "global", "policy", "other"],
}[TASK]
label2id = {label: i for i, label in enumerate(LABELS)}
print(TASK, LABELS, "->", MODEL_REPO)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()   # paste a token with WRITE access (huggingface.co -> Settings -> Access Tokens)

In [ ]:
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download

folder = snapshot_download(DATA_REPO, repo_type="dataset")
df = pd.read_csv(f"{folder}/{TASK}.csv")
df["y"] = df["label"].map(label2id)
assert df["y"].notna().all(), "unknown labels in the data"

train, val, test = (df[df["split"] == s].reset_index(drop=True) for s in ("train", "validation", "test"))
print(len(train), "train |", len(val), "validation |", len(test), "test")
print(df.groupby(["split", "label"]).size().unstack(fill_value=0))

In [ ]:
import copy
import random

import torch
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)            # must say cuda; if not, fix the runtime type and restart

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=dict(enumerate(LABELS)),
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(device)

# Rare classes get larger weights, because accuracy would reward always predicting the biggest class.
weights = compute_class_weight("balanced", classes=np.arange(len(LABELS)), y=train["y"].values)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
print("class weights:", dict(zip(LABELS, weights.round(2))))

In [ ]:
def batches(frame, batch_size, shuffle):
    order = np.random.permutation(len(frame)) if shuffle else np.arange(len(frame))
    for start in range(0, len(order), batch_size):
        part = frame.iloc[order[start : start + batch_size]]
        enc = tokenizer(part["text"].tolist(), padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(device)
        yield enc, torch.tensor(part["y"].values, dtype=torch.long).to(device)


@torch.no_grad()
def predict_proba(texts, batch_size=64):
    model.eval()
    out = []
    for start in range(0, len(texts), batch_size):
        enc = tokenizer(texts[start : start + batch_size], padding=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt").to(device)
        out.append(torch.softmax(model(**enc).logits.float(), dim=-1).cpu().numpy())
    return np.vstack(out)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = EPOCHS * ((len(train) + BATCH - 1) // BATCH)
scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
use_cuda = device == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_cuda)

best_f1, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    running = 0.0
    for inputs, y in batches(train, BATCH, shuffle=True):
        optimizer.zero_grad()
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
            logits = model(**inputs).logits
        loss = loss_fn(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()
    val_pred = predict_proba(val["text"].tolist()).argmax(axis=1)
    f1 = f1_score(val["y"], val_pred, average="macro")
    print(f"epoch {epoch}: train loss {running / (len(train) / BATCH):.3f} | validation macro-F1 {f1:.3f}")
    if f1 > best_f1:
        best_f1 = f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print("best validation macro-F1:", round(best_f1, 3))

In [ ]:
for name, part in (("validation 2019-2021", val), ("test 2022-2026 (weak labels only)", test)):
    pred = predict_proba(part["text"].tolist()).argmax(axis=1)
    print(f"=== {name} ===")
    print(classification_report(part["y"], pred, labels=list(range(len(LABELS))),
                                target_names=LABELS, digits=3, zero_division=0))
    print(pd.DataFrame(confusion_matrix(part["y"], pred, labels=list(range(len(LABELS)))),
                       index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS]))

In [ ]:
model.push_to_hub(MODEL_REPO, private=True)
tokenizer.push_to_hub(MODEL_REPO, private=True)
print("Saved privately to https://huggingface.co/" + MODEL_REPO)

In [ ]:
# Predict every sentence (training sentences included, so treat those as in-sample).
everything = pd.read_csv(f"{folder}/sentences_all.csv")
probs = predict_proba(everything["text"].tolist())
out = everything[["sentence_id"]].copy()
for i, label in enumerate(LABELS):
    out[f"p_{label}"] = probs[:, i].round(5)
out.to_csv(f"predictions_{TASK}.csv", index=False)
print(out.shape)

try:
    from google.colab import files
    files.download(f"predictions_{TASK}.csv")
except ImportError:
    print("Not in Colab: the file is saved next to the notebook.")